# LumenY — Fair Model Comparison: V1 vs V2 vs V3

**All models evaluated on the same test set (2024-07 to 2025-12) using identical PnL/Sharpe calculations.**

| Model | Description | Target |
|-------|------------|--------|
| **V1** | Vol filter + Direction (all features, top 50% vol training) | 1H return |
| **V2** | Vol filter + Direction (curated features, top 33% vol training) | 1H return |
| **V3** | Sustained Move — binary 4H direction with hold-until-flip | 4H return |

**Fair comparison:** All strategies produce an hourly PnL stream (what you earn each hour). Sharpe is computed identically on these hourly streams.

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

FEATURES_DIR = Path('../backend/data/features')
MODELS_DIR   = Path('../backend/models_4')
TRAIN_END    = '2024-06-30'

print('Fair Model Comparison: V1 vs V2 vs V3')
print(f'Training cutoff: {TRAIN_END}')

## 1. Load Data & Models

In [ ]:
df = pd.read_parquet(FEATURES_DIR / 'all_pairs_features_labels.parquet')

label_cols   = [c for c in df.columns if c.startswith('label_')]
drop_cols    = label_cols + ['pair']
feature_cols = [c for c in df.columns if c not in drop_cols]

df_test = df[df.index > TRAIN_END].copy()

# Remove rows where both 1H and 4H labels are available
valid = df_test['label_1H'].notna() & df_test['label_4H'].notna()
df_test = df_test[valid]

print(f'Test set: {len(df_test):,} rows ({df_test.index.min().date()} to {df_test.index.max().date()})')
print(f'Pairs: {sorted(df_test["pair"].unique())}')

actual_return_1h = df_test['label_1H'].values

# ── Load all models ──
# V1
v1_vol_bundle = joblib.load(MODELS_DIR / 'vol_model.joblib')
v1_dir_bundle = joblib.load(MODELS_DIR / 'dir_model.joblib')
v1_vol_model  = v1_vol_bundle['model']
v1_dir_model  = v1_dir_bundle['model']
v1_vol_pct    = v1_vol_bundle['vol_percentiles']
v1_feature_cols = v1_vol_bundle['feature_cols']

# V2
v2_dir_bundle = joblib.load(MODELS_DIR / 'dir_v2' / 'dir_v2_pooled.joblib')
v2_dir_model  = v2_dir_bundle['model']
v2_feature_cols = v2_dir_bundle['feature_cols']

# V3 (sustained move)
v3_bundle = joblib.load(MODELS_DIR / 'sustained_move' / 'sustained_move_model.joblib')
v3_model  = v3_bundle['model']
v3_feature_cols = v3_bundle['feature_cols']

print(f'\nV1 vol model: {v1_vol_bundle["type"]}')
print(f'V1 dir model: {v1_dir_bundle["type"]}, features: {len(v1_feature_cols)}')
print(f'V2 dir model: {v2_dir_bundle["type"]}, features: {len(v2_feature_cols)}')
print(f'V3 model:     {v3_bundle["type"]}, features: {len(v3_feature_cols)}')

## 2. Generate Predictions

In [ ]:
# ── V1 predictions ──
X_test_v1 = df_test[v1_feature_cols].ffill().fillna(0)
v1_vol_preds = v1_vol_model.predict(X_test_v1)
v1_dir_probs = v1_dir_model.predict_proba(X_test_v1)[:, 1]  # P(UP)
v1_dir = np.where(v1_dir_probs > 0.5, 1, -1)
v1_conf = np.where(v1_dir_probs > 0.5, v1_dir_probs, 1 - v1_dir_probs)

# ── V2 predictions (uses same vol model as V1) ──
X_test_v2 = df_test[v2_feature_cols].ffill().fillna(0)
v2_dir_probs = v2_dir_model.predict_proba(X_test_v2)[:, 1]
v2_dir = np.where(v2_dir_probs > 0.5, 1, -1)
v2_conf = np.where(v2_dir_probs > 0.5, v2_dir_probs, 1 - v2_dir_probs)

# ── V3 predictions ──
X_test_v3 = df_test[v3_feature_cols].ffill().fillna(0)
v3_probs = v3_model.predict_proba(X_test_v3)[:, 1]  # P(UP)
v3_dir = np.where(v3_probs > 0.5, 1, -1)
v3_conf = np.where(v3_probs > 0.5, v3_probs, 1 - v3_probs)

print('Predictions generated for all models.')
print(f'V1 dir balance: UP={( v1_dir == 1).sum():,} DOWN={(v1_dir == -1).sum():,}')
print(f'V2 dir balance: UP={(v2_dir == 1).sum():,} DOWN={(v2_dir == -1).sum():,}')
print(f'V3 dir balance: UP={(v3_dir == 1).sum():,} DOWN={(v3_dir == -1).sum():,}')

## 3. Build Hourly PnL Streams

For fair comparison, every strategy produces an **hourly PnL series** — for each hour in the test set, what did the strategy earn?

- **V1/V2 (per-bar):** If the hour passes the vol+conf filter, PnL = `pred_dir × actual_1H_return`. Otherwise 0 (flat).
- **V3 (hold-until-flip):** If in a position that hour, PnL = `position_dir × actual_1H_return`. Otherwise 0.

Then Sharpe = `mean(hourly_pnl) / std(hourly_pnl) × sqrt(252×24)` for all strategies.

In [ ]:
def hourly_pnl_perbar(pred_dir, pred_vol, actual_ret, vol_threshold, conf, conf_threshold):
    """V1/V2 per-bar strategy: trade every hour that passes vol+conf filter."""
    mask = (pred_vol > vol_threshold) & (conf >= conf_threshold)
    pnl = np.zeros(len(pred_dir))
    pnl[mask] = pred_dir[mask] * actual_ret[mask]
    return pnl


def hourly_pnl_holdflip(signal_dir, signal_conf, actual_ret, pairs, index,
                         entry_conf=0.55, hold_conf=0.50, max_hold=8, cooldown=1):
    """V3 hold-until-flip strategy: returns hourly PnL array."""
    pnl = np.zeros(len(signal_dir))
    
    df_tmp = pd.DataFrame({
        'signal_dir': signal_dir,
        'signal_conf': signal_conf,
        'actual_return_1h': actual_ret,
        'pair': pairs,
    }, index=index)
    
    for pair in sorted(df_tmp['pair'].unique()):
        pair_data = df_tmp[df_tmp['pair'] == pair].sort_index()
        pair_iloc = [df_tmp.index.get_loc(idx) for idx in pair_data.index]
        
        position = None
        cooldown_remaining = 0
        
        for i, global_idx in enumerate(pair_iloc):
            row = pair_data.iloc[i]
            
            if position is not None:
                position['bars_held'] += 1
                pnl[global_idx] = position['dir'] * row['actual_return_1h']
                
                same_dir = (row['signal_dir'] == position['dir'])
                conf_ok  = (row['signal_conf'] >= hold_conf)
                max_reached = (position['bars_held'] >= max_hold)
                
                if not same_dir or not conf_ok or max_reached:
                    position = None
                    cooldown_remaining = cooldown
            else:
                if cooldown_remaining > 0:
                    cooldown_remaining -= 1
                    continue
                if row['signal_conf'] >= entry_conf:
                    position = {
                        'dir': row['signal_dir'],
                        'bars_held': 0,
                    }
    
    return pnl


def compute_metrics(hourly_pnl):
    """Compute metrics from hourly PnL array."""
    active = hourly_pnl != 0
    n_active = active.sum()
    total_pnl = hourly_pnl.sum()
    
    if n_active == 0:
        return {'active_hours': 0, 'total_pnl': 0, 'sharpe': 0, 'win_rate': 0, 'avg_pnl_active': 0}
    
    # Sharpe on ALL hours (including flat hours — this is what you actually experience)
    sharpe_all = (hourly_pnl.mean() / hourly_pnl.std()) * np.sqrt(252 * 24) if hourly_pnl.std() > 0 else 0
    
    # Sharpe on active hours only (for comparing signal quality)
    active_pnl = hourly_pnl[active]
    sharpe_active = (active_pnl.mean() / active_pnl.std()) * np.sqrt(252 * 24) if active_pnl.std() > 0 else 0
    
    win_rate = (active_pnl > 0).mean()
    avg_pnl_active = active_pnl.mean()
    
    return {
        'active_hours': n_active,
        'active_pct': n_active / len(hourly_pnl) * 100,
        'total_pnl': total_pnl,
        'sharpe_all': sharpe_all,
        'sharpe_active': sharpe_active,
        'win_rate': win_rate,
        'avg_pnl_active': avg_pnl_active,
    }

print('Helper functions defined.')

## 4. Strategy Comparison Table

In [ ]:
pairs_arr = df_test['pair'].values
test_index = df_test.index

strategies = {}

# ── V1 configs ──
for vp, cf, label in [
    (50, 0.50, 'V1: Top50% vol'),
    (67, 0.50, 'V1: Top33% vol'),
    (90, 0.50, 'V1: Top10% vol'),
    (90, 0.55, 'V1: Top10% vol + conf>=55%'),
]:
    pnl = hourly_pnl_perbar(v1_dir, v1_vol_preds, actual_return_1h, v1_vol_pct[vp], v1_conf, cf)
    strategies[label] = pnl

# ── V2 configs (same vol model, different dir model) ──
for vp, cf, label in [
    (50, 0.50, 'V2: Top50% vol'),
    (67, 0.50, 'V2: Top33% vol'),
    (90, 0.50, 'V2: Top10% vol'),
    (90, 0.55, 'V2: Top10% vol + conf>=55%'),
]:
    pnl = hourly_pnl_perbar(v2_dir, v1_vol_preds, actual_return_1h, v1_vol_pct[vp], v2_conf, cf)
    strategies[label] = pnl

# ── V3 configs ──
for ec, hc, label in [
    (0.50, 0.50, 'V3: conf>=0.50'),
    (0.54, 0.50, 'V3: conf>=0.54'),
    (0.56, 0.50, 'V3: conf>=0.56'),
    (0.58, 0.50, 'V3: conf>=0.58'),
    (0.60, 0.50, 'V3: conf>=0.60'),
]:
    pnl = hourly_pnl_holdflip(v3_dir, v3_conf, actual_return_1h, pairs_arr, test_index,
                               entry_conf=ec, hold_conf=hc, max_hold=8, cooldown=1)
    strategies[label] = pnl

# ── Print comparison table ──
print(f'{"Strategy":<35} {"Active hrs":>10} {"Active%":>8} {"Win%":>7} {"Total PnL":>11} {"Sharpe(all)":>12} {"Sharpe(act)":>12}')
print('=' * 100)

prev_model = ''
for label, pnl in strategies.items():
    m = compute_metrics(pnl)
    model = label.split(':')[0]
    if model != prev_model and prev_model != '':
        print('-' * 100)
    prev_model = model
    
    print(f'{label:<35} {m["active_hours"]:>10,} {m["active_pct"]:>7.1f}% {m["win_rate"]:>6.1%} {m["total_pnl"]:>11.4f} {m["sharpe_all"]:>12.2f} {m["sharpe_active"]:>12.2f}')

## 5. Equity Curves — Best Config per Model

In [ ]:
# Pick best config per model by Sharpe(all hours)
best_per_model = {}
for label, pnl in strategies.items():
    model = label.split(':')[0]
    m = compute_metrics(pnl)
    if model not in best_per_model or m['sharpe_all'] > best_per_model[model][1]['sharpe_all']:
        best_per_model[model] = (label, m, pnl)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.patch.set_facecolor('#080c14')

colors = {'V1': '#4fc3f7', 'V2': '#81c784', 'V3': '#ffb74d'}

for ax, (model, (label, m, pnl)) in zip(axes, best_per_model.items()):
    ax.set_facecolor('#080c14')
    
    cum_pnl = pd.Series(pnl, index=test_index).cumsum()
    ax.plot(cum_pnl.index, cum_pnl.values, color=colors[model], linewidth=2)
    ax.axhline(0, color=(1,1,1,0.2), linewidth=1, linestyle='--')
    
    ax.set_title(f'{label}\nSharpe={m["sharpe_all"]:.2f}, PnL={m["total_pnl"]:.2f}, Active={m["active_pct"]:.0f}%',
                 color='white', fontsize=10)
    ax.tick_params(colors='white')
    ax.set_ylabel('Cumulative log return', color='white', fontsize=9)
    for spine in ax.spines.values(): spine.set_edgecolor('#1a2332')

plt.suptitle('Best Config per Model — Equity Curves (UNSEEN test set)', color='white', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Overlay — All Best Configs on One Chart

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))
fig.patch.set_facecolor('#080c14')
ax.set_facecolor('#080c14')

for model, (label, m, pnl) in best_per_model.items():
    cum_pnl = pd.Series(pnl, index=test_index).cumsum()
    ax.plot(cum_pnl.index, cum_pnl.values, color=colors[model], linewidth=2,
            label=f'{label} (Sharpe={m["sharpe_all"]:.2f}, PnL={m["total_pnl"]:.2f})')

ax.axhline(0, color=(1,1,1,0.2), linewidth=1, linestyle='--')
ax.legend(facecolor='#0d1117', edgecolor='#1a2332', labelcolor='white', fontsize=10)
ax.set_title('V1 vs V2 vs V3 — Best Configs Overlay (UNSEEN test set)', color='white', fontsize=13)
ax.set_ylabel('Cumulative log return', color='white')
ax.tick_params(colors='white')
for spine in ax.spines.values(): spine.set_edgecolor('#1a2332')

plt.tight_layout()
plt.show()

## 7. Per-Pair Breakdown — Best Config per Model

In [ ]:
for model, (label, m, pnl) in best_per_model.items():
    print(f'\n{"=" * 70}')
    print(f'{label}')
    print(f'{"=" * 70}')
    
    pnl_series = pd.Series(pnl, index=test_index)
    
    print(f'{"Pair":<10} {"Active hrs":>10} {"Win%":>7} {"Total PnL":>11} {"Sharpe(act)":>12}')
    print('-' * 55)
    
    for pair in sorted(df_test['pair'].unique()):
        pair_mask = df_test['pair'].values == pair
        pair_pnl = pnl[pair_mask]
        active = pair_pnl != 0
        n_active = active.sum()
        
        if n_active == 0:
            print(f'{pair:<10} {0:>10} {"N/A":>7} {0:>11.4f} {"N/A":>12}')
            continue
        
        active_pnl = pair_pnl[active]
        win_r = (active_pnl > 0).mean()
        total = pair_pnl.sum()
        sharpe = (active_pnl.mean() / active_pnl.std()) * np.sqrt(252 * 24) if active_pnl.std() > 0 else 0
        
        print(f'{pair:<10} {n_active:>10,} {win_r:>6.1%} {total:>11.4f} {sharpe:>12.2f}')

## 8. Monthly PnL Heatmap — Best Config per Model

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.patch.set_facecolor('#080c14')

for ax, (model, (label, m, pnl)) in zip(axes, best_per_model.items()):
    ax.set_facecolor('#080c14')
    
    pnl_series = pd.Series(pnl, index=test_index)
    monthly = pnl_series.resample('ME').sum()
    
    bar_colors = ['#4caf50' if v > 0 else '#f44336' for v in monthly.values]
    ax.bar(range(len(monthly)), monthly.values, color=bar_colors, alpha=0.8)
    ax.set_xticks(range(len(monthly)))
    ax.set_xticklabels([d.strftime('%Y-%m') for d in monthly.index], rotation=45, fontsize=7, color='white')
    ax.axhline(0, color=(1,1,1,0.3), linewidth=1)
    ax.set_title(f'{model} — Monthly PnL', color='white', fontsize=10)
    ax.tick_params(colors='white')
    for spine in ax.spines.values(): spine.set_edgecolor('#1a2332')
    
    pos_months = (monthly > 0).sum()
    tot_months = len(monthly)
    ax.text(0.02, 0.95, f'{pos_months}/{tot_months} months positive', transform=ax.transAxes,
            color='white', fontsize=8, va='top')

plt.suptitle('Monthly PnL — Best Config per Model', color='white', fontsize=13)
plt.tight_layout()
plt.show()

## 9. Summary

In [ ]:
print('=' * 80)
print('FAIR MODEL COMPARISON — SUMMARY')
print('=' * 80)
print(f'\nTest period: {df_test.index.min().date()} to {df_test.index.max().date()} (UNSEEN)')
print(f'Total test hours: {len(df_test):,}')
print(f'\nAll Sharpe values computed identically: mean(hourly_pnl) / std(hourly_pnl) × sqrt(252×24)')

print(f'\n{"Model":<10} {"Best Config":<30} {"Sharpe":>8} {"Total PnL":>11} {"Active%":>8} {"Win%":>7}')
print('-' * 80)

for model, (label, m, pnl) in best_per_model.items():
    config = label.split(': ')[1]
    print(f'{model:<10} {config:<30} {m["sharpe_all"]:>8.2f} {m["total_pnl"]:>11.4f} {m["active_pct"]:>7.1f}% {m["win_rate"]:>6.1%}')

# Winner
winner = max(best_per_model.items(), key=lambda x: x[1][1]['sharpe_all'])
print(f'\nWinner by Sharpe(all hours): {winner[1][0]}')

## 10. Economic Value (EV%) — Live Trading Projections

**EV% = (win_rate × avg_|move|) - ((1 - win_rate) × avg_|move|) - spread_cost**

Simplifies to: `EV% = (2 × win_rate - 1) × avg_|move| - spread_cost`

This is what you'd actually earn per trade after spread. Same formula as the paper trading dashboard.

- `avg_|move|` = average absolute 1H return on active hours (what the market moves when you trade)
- `spread_cost` = 2.8 pips ≈ 0.00028 (average forex spread)
- Positive EV% = profitable edge after costs

In [ ]:
AVG_SPREAD = 0.00028  # ~2.8 pips in raw log return units

rows = []
ev_data = {}  # store raw EV per strategy for best-config selection

for label, pnl in strategies.items():
    active = pnl != 0
    n_active = active.sum()
    if n_active == 0:
        continue
    
    active_pnl = pnl[active]
    win_rate = (active_pnl > 0).mean()
    avg_abs_move = np.abs(actual_return_1h[active]).mean()
    
    # EV per trade (raw log return units)
    ev_per_trade = (2 * win_rate - 1) * avg_abs_move - AVG_SPREAD
    
    # Annualized: EV per trade × trades per year
    test_days = (df_test.index.max() - df_test.index.min()).days
    test_years = test_days / 365.25
    trades_per_year = n_active / test_years
    ev_annual = ev_per_trade * trades_per_year
    
    ev_data[label] = {
        'ev_per_trade': ev_per_trade,
        'win_rate': win_rate,
        'avg_abs_move': avg_abs_move,
        'trades_per_year': trades_per_year,
        'ev_annual': ev_annual,
        'pnl': pnl,
    }
    
    rows.append({
        'Strategy': label,
        'Win%': f'{win_rate:.1%}',
        'Avg |move|': f'{avg_abs_move:.6f}',
        'Spread cost': f'{AVG_SPREAD:.6f}',
        'EV/trade': f'{ev_per_trade:.6f}',
        'EV/trade (bps)': f'{ev_per_trade * 10000:.2f}',
        'Trades/yr': f'{trades_per_year:,.0f}',
        'Annual EV': f'{ev_annual:.4f}',
        'Annual EV (bps)': f'{ev_annual * 10000:.0f}',
        'Edge?': '✅' if ev_per_trade > 0 else '❌',
    })

ev_df = pd.DataFrame(rows)
print('Economic Value (EV%) — All Configurations')
print(f'Spread cost: {AVG_SPREAD:.5f} ({AVG_SPREAD*10000:.1f} bps)\n')
print(ev_df.to_string(index=False))

# ── Best config per model by EV/trade (not Sharpe) ──
print('\n' + '=' * 80)
print('BEST CONFIG PER MODEL — BY EV/TRADE')
print('=' * 80)

best_ev_per_model = {}
for label, d in ev_data.items():
    model = label.split(':')[0]
    if model not in best_ev_per_model or d['ev_per_trade'] > best_ev_per_model[model][1]['ev_per_trade']:
        best_ev_per_model[model] = (label, d)

for model, (label, d) in best_ev_per_model.items():
    edge = 'POSITIVE EDGE ✅' if d['ev_per_trade'] > 0 else 'NO EDGE ❌'
    print(f'\n{label}')
    print(f'  Win rate:      {d["win_rate"]:.1%}')
    print(f'  Avg |move|:    {d["avg_abs_move"]:.6f} ({d["avg_abs_move"]*10000:.2f} bps)')
    print(f'  EV per trade:  {d["ev_per_trade"]:.6f} ({d["ev_per_trade"]*10000:.2f} bps) → {edge}')
    print(f'  Trades/year:   {d["trades_per_year"]:,.0f}')
    print(f'  Annual EV:     {d["ev_annual"]:.4f} ({d["ev_annual"]*10000:.0f} bps)')